# Logistic Regression Workflow

**Project question:** How do I fit and interpret a binary-response regression model?

By the end of this notebook, you should be able to:

- fit a logistic model with explicit categorical reference levels
- translate log-odds coefficients into odds ratios with confidence intervals
- produce conditional predicted probabilities without causal overclaiming

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
df = pd.read_csv(DATA / 'simulated_churn.csv')
formula = (
    'churn ~ tenure_months + monthly_charge + support_tickets + usage_gb '
    '+ C(contract, Treatment(reference="month_to_month")) '
    '+ C(autopay, Treatment(reference="no")) + satisfaction'
)
model = smf.logit(formula, data=df).fit(disp=False)
model.summary()

The categorical coefficients compare with `month_to_month` for contract and `no` for autopay. Maximum likelihood estimates the coefficients; the reported standard errors and intervals rely on the fitted-model assumptions.

In [ ]:
interval = model.conf_int()
coef = model.params.to_frame('log_odds_coef')
coef['odds_ratio'] = np.exp(coef['log_odds_coef'])
coef['odds_ratio_ci_low'] = np.exp(interval[0])
coef['odds_ratio_ci_high'] = np.exp(interval[1])
coef

In [ ]:
ticket_or = coef.loc['support_tickets']
print(
    'Holding the listed predictors constant, one additional support ticket '
    f"multiplies the estimated churn odds by {ticket_or['odds_ratio']:.2f} "
    f"(95% CI {ticket_or['odds_ratio_ci_low']:.2f} to {ticket_or['odds_ratio_ci_high']:.2f})."
)

An odds ratio is multiplicative on odds, not an absolute probability increase. The probability change depends on all predictors and the starting probability.

In [ ]:
new_customers = pd.DataFrame({
    'tenure_months': [6, 36],
    'monthly_charge': [85, 60],
    'support_tickets': [3, 0],
    'usage_gb': [60, 110],
    'contract': ['month_to_month', 'two_year'],
    'autopay': ['no', 'yes'],
    'satisfaction': [5.0, 8.5]
})
new_customers['predicted_churn_probability'] = model.predict(new_customers)
new_customers

**Interpretation:** These predictions are conditional on the specified profiles and the synthetic data-generating relationship. They are not treatment effects: changing a support-ticket count in the table does not prove that causing another ticket would change churn.